# HTMX Integration Example

> Real-time progress tracking with HTMX using conditional polling and granular cell-level updates. Demonstrates server-side rendering with optimized polling strategies that automatically start/stop based on job status, minimizing unnecessary requests while maintaining responsive UI updates.

In [1]:
from fasthtml.common import *
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_sizes
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CUPCAKE)

# Initialize
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Sample tasks
def file_processing_task():
    from tqdm import tqdm
    import time
    
    files = [f"file_{i}.txt" for i in range(50)]
    processed = []
    
    for file in tqdm(files, desc="Processing files"):
        time.sleep(0.05)  # Simulate file processing
        processed.append(file)
    
    return processed

def data_analysis_task():
    from tqdm import tqdm
    import time
    
    # Load data
    for _ in tqdm(range(30), desc="Loading data"):
        time.sleep(0.02)
    
    # Analyze
    for _ in tqdm(range(50), desc="Analyzing"):
        time.sleep(0.03)
    
    # Generate report
    for _ in tqdm(range(20), desc="Generating report"):
        time.sleep(0.02)
    
    return "Analysis complete"

In [5]:
# HTMX-powered UI with helper functions for conditional polling and cell-level updates
def render_task_buttons(disabled=False, oob_swap=False):
    """Render task buttons with appropriate state
    
    Args:
        disabled: Whether buttons should be disabled
        oob_swap: Whether to enable out-of-band swap for HTMX
    """
    btn_classes_files = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else "",
        m.r(2)
    )
    btn_classes_analysis = combine_classes(
        btn,
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'id': 'task-buttons',
        'cls': str(m.b(4))
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = "true"
    
    return Div(
        Button(
            "Process Files",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "files"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_files,
            disabled=disabled,
            id="process-files-btn"
        ),
        Button(
            "Run Analysis",
            hx_post="/start_task" if not disabled else None,
            hx_vals='{"task_type": "analysis"}' if not disabled else None,
            hx_target="#progress-container" if not disabled else None,
            hx_swap="innerHTML" if not disabled else None,
            cls=btn_classes_analysis,
            disabled=disabled,
            id="run-analysis-btn"
        ),
        **kwargs
    )

def render_progress_bar(value, max_value="100", color=None, width=None):
    """Render a progress bar element
    
    Args:
        value: The current progress value
        max_value: The maximum value (default: "100")
        color: The progress color class (e.g., progress_colors.primary)
        width: The width class (e.g., w.full)
    """
    classes = [progress]
    if color:
        classes.append(color)
    if width:
        classes.append(width)
    
    return Progress(
        value=str(value),
        max=max_value,
        cls=combine_classes(*classes)
    )

def render_overall_progress(job_id, progress_value, enable_polling=False, oob_swap=False):
    """Render the overall progress section with conditional polling
    
    Args:
        job_id: The job ID
        progress_value: Current progress value (0-100)
        enable_polling: Whether to enable polling for updates
        oob_swap: Whether to enable out-of-band swap
    """
    kwargs = {
        'id': f'overall-progress-{job_id}'
    }
    
    if enable_polling:
        kwargs['hx_get'] = f"/overall_progress?job_id={job_id}"
        kwargs['hx_trigger'] = "load delay:200ms"
        kwargs['hx_swap'] = "outerHTML"
    
    if oob_swap:
        kwargs['hx_swap_oob'] = "true"
    
    return Div(
        P(f"Overall: {progress_value:.1f}%", 
          cls=str(font_weight.bold),
          id=f"overall-text-{job_id}"),
        render_progress_bar(progress_value, 
                          color=progress_colors.primary, 
                          width=w.full),
        **kwargs
    )

def render_task_bar(job_id, bar_id, bar_data, enable_polling=False, oob_swap=False):
    """Render an individual task progress bar with conditional polling
    
    Args:
        job_id: The job ID
        bar_id: The bar ID
        bar_data: Bar data with description, progress, current, total
        enable_polling: Whether to enable polling for updates
        oob_swap: Whether to enable out-of-band swap
    """
    kwargs = {
        'id': f'bar-{job_id}-{bar_id}',
        'cls': str(m.b(2))
    }
    
    if enable_polling:
        kwargs['hx_get'] = f"/task_bar?job_id={job_id}&bar_id={bar_id}"
        kwargs['hx_trigger'] = "load delay:200ms"
        kwargs['hx_swap'] = "outerHTML"
    
    if oob_swap:
        kwargs['hx_swap_oob'] = "true"
    
    return Div(
        P(f"{bar_data.description}: {bar_data.progress:.1f}% ({bar_data.current}/{bar_data.total})", 
          cls=str(font_size.sm)),
        render_progress_bar(bar_data.progress, 
                          color=progress_colors.accent, 
                          width=w.full),
        **kwargs
    )

def render_status_badge(text, color):
    """Render a status badge
    
    Args:
        text: The status text to display
        color: The badge color class (e.g., badge_colors.success)
    """
    return Span(
        text,
        cls=combine_classes(badge, color)
    )

def render_job_row(job_id, job, enable_polling=False):
    """Render a single job row with conditional polling
    
    Args:
        job_id: The job ID
        job: The job data from monitor
        enable_polling: Whether to enable polling for this row
    """
    return Tr(
        Td(job_id[:8] + "...", id=f"job-id-cell-{job_id}"),
        Td(
            f"{job['overall_progress']:.1f}%",
            id=f"job-progress-cell-{job_id}",
            hx_get=f"/job_progress_cell?job_id={job_id}" if enable_polling and not job['completed'] else None,
            hx_trigger="load delay:500ms" if enable_polling and not job['completed'] else None,
            hx_swap="innerHTML"
        ),
        Td(
            render_status_badge(
                "Complete" if job['completed'] else "Running",
                badge_colors.success if job['completed'] else badge_colors.warning
            ),
            id=f"job-status-cell-{job_id}",
            hx_get=f"/job_status_cell?job_id={job_id}" if enable_polling and not job['completed'] else None,
            hx_trigger="load delay:500ms" if enable_polling and not job['completed'] else None,
            hx_swap="innerHTML"
        ),
        Td(str(len(job['bars'])), id=f"job-bars-cell-{job_id}"),
        id=f"job-row-{job_id}"
    )

def render_hidden_trigger(endpoint, target_id, swap="outerHTML", trigger="load"):
    """Render a hidden div that triggers an HTMX update
    
    Args:
        endpoint: The endpoint to call (e.g., "/jobs_list")
        target_id: The target element ID to update (e.g., "#jobs-container")
        swap: The HTMX swap strategy (default: "outerHTML")
        trigger: The HTMX trigger (default: "load")
    """
    from cjm_fasthtml_tailwind.utilities.layout import display_tw
    
    return Div(
        hx_get=endpoint,
        hx_trigger=trigger,
        hx_target=target_id,
        hx_swap=swap,
        cls=str(display_tw.hidden)
    )

@rt("/")
def index():
    return create_test_page(
        "HTMX Progress Demo",
        Div(
            H1("HTMX + Conditional Polling Progress Tracking", cls=combine_classes(font_size._2xl, font_weight.bold, m.b(6))),
            
            # Task selection
            Div(
                H2("Select Task", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                render_task_buttons(disabled=False),  # Start with enabled buttons
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Active jobs with conditional polling
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                Div(
                    id="jobs-container",
                    hx_get="/jobs_list",
                    hx_trigger="load",  # Only load initially, polling added conditionally
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [6]:
# HTMX endpoints with optimized cell-level updates
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'files')
    
    job_id = str(uuid.uuid4())
    
    # Select task based on type
    if task_type == 'analysis':
        task = data_analysis_task
        task_name = "Data Analysis"
    else:
        task = file_processing_task
        task_name = "File Processing"
    
    # Start the job
    runner.start(
        job_id,
        task,
        patch_kwargs={
            "min_delta_pct": 2,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Small delay to allow monitor to register the job
    import time
    time.sleep(0.05)
    
    # Get initial snapshot to render progress bars
    snapshot = monitor.snapshot(job_id)
    
    # Build initial progress display with individual polling elements
    progress_elements = []
    
    # Overall progress with its own polling
    if snapshot:
        progress_elements.append(
            render_overall_progress(job_id, snapshot['overall_progress'], enable_polling=True)
        )
        
        # Individual task bars with their own polling
        if snapshot['bars']:
            bars_container = Div(
                *[render_task_bar(job_id, bar_id, bar, enable_polling=True) 
                  for bar_id, bar in snapshot['bars'].items()],
                id=f"bars-container-{job_id}",
                cls=str(m.t(4))
            )
            progress_elements.append(bars_container)
    else:
        # If no snapshot yet, show waiting state with polling
        progress_elements.append(
            render_overall_progress(job_id, 0, enable_polling=True)
        )
    
    # Return optimized response with individual polling elements
    return Div(
        # Disable buttons via out-of-band swap
        render_task_buttons(disabled=True, oob_swap=True),
        # Progress display
        Div(
            H3(f"{task_name} Progress", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
            
            # Status
            Div(
                f"Job ID: {job_id[:8]}...",
                id=f"job-status-{job_id}",
                cls=combine_classes(alert, alert_colors.info, m.b(4))
            ),
            
            # Progress elements with individual polling
            Div(
                *progress_elements,
                id=f"progress-content-{job_id}"
            ),
            
            cls=combine_classes(card, bg_dui.base_200, p(6))
        ),
        # Hidden trigger to refresh jobs list
        render_hidden_trigger("/jobs_list", "#jobs-container", swap="innerHTML")
    )

@rt("/job_progress")
def job_progress(job_id: str):
    """Returns the current progress state for a job - simplified version for fallback"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not found or not started yet - continue polling
        return Div(
            P("Waiting for job to start...", cls=str(font_weight.bold)),
            render_progress_bar(0, color=progress_colors.primary, width=w.full),
            # Keep polling until job starts
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:100ms",
            hx_swap="outerHTML"
        )
    
    # Build individual progress bars
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            render_task_bar(job_id, bar_id, bar, enable_polling=not snapshot['completed'])
        )
    
    # Create wrapper for progress content
    content = [
        render_overall_progress(job_id, snapshot['overall_progress'], 
                               enable_polling=not snapshot['completed']),
        Div(*bars_html, id=f"bars-container-{job_id}", cls=str(m.t(4))) if bars_html else None
    ]
    
    # Check if completed
    if snapshot['completed']:
        # Check if there are any other running jobs
        all_jobs = monitor.all()
        has_running = any(not job['completed'] for job in all_jobs.values() if job)
        
        # Return final state with conditionally re-enabled buttons
        return Div(
            *content,
            # Re-enable buttons if no other jobs are running via OOB
            render_task_buttons(disabled=has_running, oob_swap=True),
            # Update the status alert
            Div(
                "Task completed!",
                id=f"job-status-{job_id}",
                cls=combine_classes(alert, alert_colors.success, m.b(4)),
                hx_swap_oob="true"
            ),
            # Trigger jobs list refresh
            render_hidden_trigger("/jobs_list", "#jobs-container", swap="innerHTML")
        )
    
    # Task is in progress - content already has polling enabled
    return Div(*content)

In [7]:
@rt("/jobs_list")
def jobs_list():
    """Jobs list with conditional polling and cell-level updates"""
    jobs = monitor.all()
    
    if not jobs:
        return P("No active jobs", cls=str(text_color.gray(500)))
    
    # Check if any jobs are running
    has_running = any(not job['completed'] for job in jobs.values())
    
    # Build rows with cell-level polling for running jobs
    rows = []
    for job_id, job in jobs.items():
        rows.append(render_job_row(job_id, job, enable_polling=has_running))
    
    # Create table
    table_elem = Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Bars")
            )
        ),
        Tbody(*rows, id="jobs-tbody"),
        cls=combine_classes(table, table_sizes.xs, w.full)
    )
    
    # Create wrapper with conditional polling for the container
    wrapper = Div(table_elem)
    
    # Add container-level polling only if jobs are running
    # This will trigger a refresh of the entire list periodically
    if has_running:
        wrapper.attrs['hx-trigger'] = "load delay:2s"
        wrapper.attrs['hx-get'] = "/jobs_list"
        wrapper.attrs['hx-swap'] = "outerHTML"
    
    return wrapper

In [8]:
# Individual cell update endpoints for precise updates
@rt("/job_progress_cell")
def job_progress_cell(job_id: str):
    """Update only the progress cell for a specific job"""
    job = monitor.all().get(job_id)
    
    if not job:
        return ""
    
    # Return the progress text
    progress_text = f"{job['overall_progress']:.1f}%"
    
    # If job is still running, continue polling for this cell
    if not job['completed']:
        return Div(
            progress_text,
            hx_get=f"/job_progress_cell?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )
    else:
        # Job completed, stop polling
        return progress_text

@rt("/job_status_cell")
def job_status_cell(job_id: str):
    """Update only the status cell for a specific job"""
    job = monitor.all().get(job_id)
    
    if not job:
        return ""
    
    # Create status badge
    status_badge = render_status_badge(
        "Complete" if job['completed'] else "Running",
        badge_colors.success if job['completed'] else badge_colors.warning
    )
    
    # If job is still running, continue polling for this cell
    if not job['completed']:
        return Div(
            status_badge,
            hx_get=f"/job_status_cell?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )
    else:
        # Job completed, stop polling and trigger button check
        return Div(
            status_badge,
            # Check if we should re-enable buttons
            render_hidden_trigger("/check_buttons", "#task-buttons", swap="outerHTML")
        )

@rt("/overall_progress")
def overall_progress(job_id: str):
    """Update only the overall progress for a specific job"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not started yet, continue polling
        return render_overall_progress(job_id, 0, enable_polling=True)
    
    # Check if completed
    if snapshot['completed']:
        # Return final progress without polling
        return Div(
            render_overall_progress(job_id, snapshot['overall_progress'], enable_polling=False),
            # Update status via OOB
            Div(
                "Task completed!",
                id=f"job-status-{job_id}",
                cls=combine_classes(alert, alert_colors.success, m.b(4)),
                hx_swap_oob="true"
            ),
            # Check buttons
            render_hidden_trigger("/check_buttons", "#task-buttons", swap="outerHTML"),
            # Refresh jobs list
            render_hidden_trigger("/jobs_list", "#jobs-container", swap="innerHTML")
        )
    
    # Still running, continue polling
    return render_overall_progress(job_id, snapshot['overall_progress'], enable_polling=True)

@rt("/task_bar")
def task_bar(job_id: str, bar_id: str):
    """Update only a specific task bar"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot or bar_id not in snapshot['bars']:
        return ""
    
    bar = snapshot['bars'][bar_id]
    
    # If job is completed or this specific bar is done (progress >= 100), stop polling
    if snapshot['completed'] or bar.progress >= 100:
        return render_task_bar(job_id, bar_id, bar, enable_polling=False)
    
    # Continue polling for this bar
    return render_task_bar(job_id, bar_id, bar, enable_polling=True)

@rt("/check_buttons")
def check_buttons():
    """Check if buttons should be re-enabled based on running jobs"""
    all_jobs = monitor.all()
    has_running = any(not job['completed'] for job in all_jobs.values())
    
    # Return buttons with appropriate state
    return render_task_buttons(disabled=has_running)

In [9]:
# Alternative polling endpoint for HTMX (instead of SSE)
@rt("/poll")
def poll(job_id: str):
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Return updated progress component
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=str(font_size.sm)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Include polling trigger only if not complete
    poll_trigger = Div(
        hx_get=f"/poll?job_id={job_id}",
        hx_trigger="load delay:500ms",
        hx_swap="outerHTML"
    ) if not snapshot['completed'] else None
    
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
        ),
        *bars_html,
        Div("Complete!", cls=combine_classes(alert, alert_colors.success)) if snapshot['completed'] else None,
        poll_trigger,
        id=f"poll-{job_id}"
    )

In [10]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [11]:
# Stop server when done
server.stop()